# 🧠 GRPO — math reasoning with verifiable reward

Uses `GSM8KRewardSignal` (carried over from v2.0) and the auto-resolved Qwen template. Pick any `DOMAIN_NAME` you want — the notebook bootstraps the YAML if it doesn't exist.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('.'))

DOMAIN_NAME = 'ai_llm_math'   # ← pick any name; e.g. 'asset_integrity_math', 'legal_reasoning'
BASE_MODEL  = 'Qwen/Qwen2.5-1.5B-Instruct'

from app.config_loader import create_domain_config, domain_config_exists, load_domain_config

if not domain_config_exists(DOMAIN_NAME):
    create_domain_config(
        name=DOMAIN_NAME,
        system_prompt='You are a careful mathematical reasoner. Always show your working and end with the final answer in \\boxed{}.',
        constitution=[
            'Show every step of the calculation.',
            'End the response with the final answer inside \\boxed{}.',
        ],
    )
    print(f'✅ Created configs/domains/{DOMAIN_NAME}.yaml')

config = load_domain_config(DOMAIN_NAME)

In [ ]:
from app.trainers import AgnosticGRPOTrainer
from app.trainers.reward_signals import GSM8KRewardSignal

trainer = AgnosticGRPOTrainer(
    config=config,
    base_model_id=BASE_MODEL,
    hf_dataset_config={
        'repo_id': 'openai/gsm8k',
        'split': 'train',
        'subset': 'main',
        'input_column': 'question',
        'output_column': 'answer',
        'max_samples': 500,
    },
    reward_signal=GSM8KRewardSignal(),
)
result = trainer.train()
result